# Split into Train/Validation/Test

In [1]:
import os
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# Load Data
DATASET_DIR = "/mnt/e/linux_projects/MIA/task6/task6_2/data"  # Update path if necessary
df = pd.read_csv(os.path.join(DATASET_DIR, "captions.txt"))
df["caption"] = df["caption"].astype(str).str.strip()




# 15% Test
gss_test = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_val_idx, test_idx = next(gss_test.split(df, groups=df["image"]))

train_val_df = df.iloc[train_val_idx]
test_df = df.iloc[test_idx].reset_index(drop=True)

# remaining 85% (Train 70% and Val 15%)
gss_val = GroupShuffleSplit(n_splits=1, test_size=0.1765, random_state=42)  # 17.65% of the 85% is 15% of the original data set
train_idx, val_idx = next(gss_val.split(train_val_df, groups=train_val_df["image"]))

train_df = train_val_df.iloc[train_idx].reset_index(drop=True)
val_df = train_val_df.iloc[val_idx].reset_index(drop=True)


# printing results
print("=== Split Summary ===")
print(f"Train Set: {len(train_df)} captions across {train_df['image'].nunique()} images")
print(f"Val Set:   {len(val_df)} captions across {val_df['image'].nunique()} images")
print(f"Test Set:  {len(test_df)} captions across {test_df['image'].nunique()} images")

# Ensure no image appears in more than one set
train_imgs = set(train_df["image"])
val_imgs = set(val_df["image"])
test_imgs = set(test_df["image"])

# assert len(train_imgs.intersection(val_imgs)) == 0, "Leakage between Train & Val!"
# assert (len(train_imgs.intersection(test_imgs)) == 0), "Leakage between Train & Test!"
# assert len(val_imgs.intersection(test_imgs)) == 0, "Leakage between Val & Test!"

# print("\n No data leakage detected across splits!")

=== Split Summary ===
Train Set: 28315 captions across 5663 images
Val Set:   6070 captions across 1214 images
Test Set:  6070 captions across 1214 images


# Caption preprocessing

In [ ]:
import collections
import re
import torch
from torch.nn.utils.rnn import pad_sequence


class Vocabulary:

    def __init__(self, freq_threshold=2):
        self.freq_threshold = freq_threshold

        # Special tokens
        self.pad_token = "<PAD>"
        self.sos_token = "<SOS>" # Start of Sentence
        self.eos_token = "<EOS>" # End of Sentence
        self.unk_token = "<UNK>" # unknown words (OOV)

        self.itos = { 0: self.pad_token,
                     1: self.sos_token,
                     2: self.eos_token,
                     3: self.unk_token}
        
        self.stoi = {v: k for k, v in self.itos.items()}

    def __len__(self):
        return len(self.itos)

    @staticmethod
    def tokenize(text):
        # Makes all text lowercase and get rid off non-alphanumeric characters
        text = text.lower()
        tokens = re.findall(r"\b\w+\b", text)
        return tokens

    def build_vocabulary(self, sentence_list):
        frequencies = collections.Counter()
        idx = len(self.itos)

        # Count frequencies
        for sentence in sentence_list:
            tokens = self.tokenize(sentence)
            frequencies.update(tokens)

        # Add words meeting threshold
        for word, count in frequencies.items():
            if count >= self.freq_threshold:
                self.stoi[word] = idx
                self.itos[idx] = word
                idx += 1

    def numericalize(self, text):
        # Converts text to numbers
        tokens = self.tokenize(text)
        numericalized = [self.stoi[self.sos_token]]

        for token in tokens:
            numericalized.append(self.stoi.get(token, self.stoi[self.unk_token]))

        numericalized.append(self.stoi[self.eos_token])
        return numericalized



# Building Vocabulary strictly on train_df

vocab = Vocabulary(freq_threshold=2)
vocab.build_vocabulary(train_df["caption"].tolist())

print(f"Vocabulary Size: {len(vocab)} unique tokens")










# -------------------------------------------------------------
sample_train_caption = train_df["caption"].iloc[0]
sample_val_caption = val_df["caption"].iloc[0]

train_numerical = vocab.numericalize(sample_train_caption)
val_numerical = vocab.numericalize(sample_val_caption)

print("\n--- Train Example ---")
print(f"Original Text:  '{sample_train_caption}'")
print(f"Numericalized:  {train_numerical}")
print(f"Decoded Back:   {[vocab.itos[i] for i in train_numerical]}")

print("\n--- Validation Example (Out-of-Vocab words become <UNK>) ---")
print(f"Original Text:  '{sample_val_caption}'")
print(f"Numericalized:  {val_numerical}")
print(f"Decoded Back:   {[vocab.itos[i] for i in val_numerical]}")

Vocabulary Size: 4431 unique tokens

--- Train Example ---
Original Text:  'A child in a pink dress is climbing up a set of stairs in an entry way .'
Numericalized:  [1, 4, 5, 6, 4, 7, 8, 9, 10, 11, 4, 12, 13, 14, 6, 15, 3, 16, 2]
Decoded Back:   ['<SOS>', 'a', 'child', 'in', 'a', 'pink', 'dress', 'is', 'climbing', 'up', 'a', 'set', 'of', 'stairs', 'in', 'an', '<UNK>', 'way', '<EOS>']

--- Validation Example (Out-of-Vocab words become <UNK>) ---
Original Text:  'A boy smiles in front of a stony wall in a city .'
Numericalized:  [1, 4, 214, 531, 6, 59, 13, 4, 3, 179, 6, 4, 208, 2]
Decoded Back:   ['<SOS>', 'a', 'boy', 'smiles', 'in', 'front', 'of', 'a', '<UNK>', 'wall', 'in', 'a', 'city', '<EOS>']


In [3]:
# add <PAD> token to make sure that all sequences will have the same length

class CaptionCollate:

    def __init__(self, pad_idx):
        self.pad_idx = pad_idx

    def __call__(self, batch):
        # batch contains tuples of (image_tensor, numericalized_caption_tensor)
        images = [item[0] for item in batch]
        captions = [item[1] for item in batch]

        images = torch.stack(images, dim=0)

        # Pad sequence along length dimension with <PAD> token ID
        padded_captions = pad_sequence(
            captions, batch_first=True, padding_value=self.pad_idx
        )

        return images, padded_captions


# Usage with PyTorch DataLoader
collate_fn = CaptionCollate(pad_idx=vocab.stoi["<PAD>"])

# Image feature extraction

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image

# Standard Image Transforms for Pretrained ResNet-50
transform = transforms.Compose([ transforms.Resize((224, 224)), transforms.ToTensor(),
                                 transforms.Normalize( mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225],),])


# ResNet-50 Encoder Class
class ResNetEncoder(nn.Module):

    def __init__(self, embed_size, use_attention=False):
        super(ResNetEncoder, self).__init__()
        self.use_attention = use_attention

        # Load pretrained ResNet-50
        resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

        # Freeze parameter gradients so we don't backprop through ResNet
        for param in resnet.parameters():
            param.requires_grad = False

        if not self.use_attention:
            # --- For Vanilla Encoder-Decoder ---
            # Remove only the last FC layer (keep AdaptiveAvgPool2d)
            modules = list(resnet.children())[:-1]
            self.resnet = nn.Sequential(*modules)
            # Linear layer to match decoder hidden/embed dimensions
            self.embed = nn.Linear(resnet.fc.in_features, embed_size)
            self.bn = nn.BatchNorm1d(embed_size, momentum=0.01)
        else:
            # --- For Attention Encoder-Decoder ---
            # Remove last 2 layers (AvgPool + FC) to keep spatial dimensions (8x8 grid)
            modules = list(resnet.children())[:-2]
            self.resnet = nn.Sequential(*modules)

    def forward(self, images):
        # Extract features
        features = self.resnet(images)  # Shape: (Batch, 2048, 7, 7) or (Batch, 2048, 1, 1)

        if not self.use_attention:
            # Flatten to (Batch, 2048)
            features = features.view(features.size(0), -1)
            # Project to embed_size: (Batch, embed_size)
            features = self.bn(self.embed(features))
        else:
            # Reshape spatial features to (Batch, 49, 2048) for Attention
            features = features.permute(0, 2, 3, 1)  # (Batch, 7, 7, 2048)
            features = features.view(
                features.size(0), -1, features.size(3)
            )  # (Batch, 49, 2048)

        return features

In [5]:
# Instantiate Encoder
EMBED_SIZE = 256
encoder = ResNetEncoder(embed_size=EMBED_SIZE, use_attention=False)
encoder.eval()  # Set to evaluation mode

# Load a sample image
img_path = "./data/Images/1000268201_693b08cb0e.jpg"  # Replace with valid image path
raw_img = Image.open(img_path).convert("RGB")

# Preprocess and add batch dimension -> (1, 3, 224, 224)
input_tensor = transform(raw_img).unsqueeze(0)

# Pass through Encoder
with torch.no_grad():
    feature_vector = encoder(input_tensor)

print(f'Feature Vector Shape: { feature_vector.shape}')
# Expected Output: torch.Size([1, 256])
print(f'Feature Vector: { feature_vector}')

Feature Vector Shape: torch.Size([1, 256])
Feature Vector: tensor([[ 0.3971,  0.1100,  0.0973,  0.2267,  0.2123, -0.1398, -0.1127,  0.1070,
         -0.1648,  0.2006, -0.2152, -0.1562,  0.0526,  0.0434, -0.0764,  0.3317,
          0.2862,  0.1578, -0.2550,  0.2841,  0.2827,  0.2082, -0.0276,  0.2996,
          0.0672,  0.0231,  0.1312, -0.3062,  0.1036,  0.1893,  0.2629,  0.0753,
         -0.0388,  0.5615,  0.0749, -0.0473,  0.0553, -0.2123,  0.1998, -0.0163,
          0.1382, -0.1983,  0.1521,  0.1605, -0.1373,  0.0645,  0.1000,  0.0693,
          0.2784, -0.1582, -0.0893,  0.1497,  0.0500, -0.1269,  0.1748,  0.0935,
         -0.4242,  0.0645, -0.3245, -0.0258,  0.2277, -0.1030, -0.1397, -0.2566,
          0.1644,  0.1089,  0.3099, -0.0140,  0.0954, -0.0794,  0.1443,  0.0176,
          0.1461,  0.3841,  0.2063, -0.2476,  0.0435, -0.1479, -0.1000,  0.0449,
          0.0843,  0.3493, -0.0365,  0.0682, -0.0455, -0.3011,  0.1388, -0.2362,
          0.0135, -0.1145,  0.0260,  0.3110, -0.09

# Encoder Decoder

In [ ]:
# import os
# import torch
# from PIL import Image
from torch.utils.data import DataLoader, Dataset


class Flickr8kDataset(Dataset):

    def __init__(self, df, images_dir, vocab, transform=None):
        self.df = df
        self.images_dir = images_dir
        self.vocab = vocab
        self.transform = transform

        self.images = self.df["image"].tolist()
        self.captions = self.df["caption"].tolist()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        caption = self.captions[index]
        img_id = self.images[index]
        img_path = os.path.join(self.images_dir, img_id)

        # Load & Transform Image
        image = Image.open(img_path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)

        # Numericalize Caption
        numericalized_caption = self.vocab.numericalize(caption)

        return image, torch.tensor(numericalized_caption)


# Assign each transformed image to a numericalized caption
train_dataset = Flickr8kDataset( df=train_df, images_dir=os.path.join(DATASET_DIR, "Images"), 
                                vocab=vocab, transform=transform,)

# DataLoader setup using the CaptionCollate from earlier
pad_idx = vocab.stoi["<PAD>"]
collate_fn = CaptionCollate(pad_idx=pad_idx)

train_loader = DataLoader( dataset=train_dataset, batch_size=32, 
                          shuffle=True, num_workers=2, collate_fn=collate_fn)

In [7]:
class DecoderRNN(nn.Module):

    def __init__(self, embed_size, hidden_size, vocab_size, num_layers=1):
        super(DecoderRNN, self).__init__()

        # Word Embedding Layer is created HERE
        self.embed = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers, batch_first=True)
        self.linear = nn.Linear(hidden_size, vocab_size)

    def forward(self, features, captions):
        # Pass token IDs through the embedding layer
        embeddings = self.embed(captions)

        # Concatenate image features as the first input token ($t_0$)
        # features shape: (batch_size, 1, embed_size)
        embeddings = torch.cat((features.unsqueeze(1), embeddings), dim=1)

        # Pass through LSTM
        hiddens, _ = self.lstm(embeddings)

        # Map LSTM outputs back to vocab dimensions for predicting next word
        outputs = self.linear(hiddens)
        return outputs

In [8]:
import torch
import torch.nn as nn

class CNNtoRNN(nn.Module):
    def __init__(self, embed_size, hidden_size, vocab_size, num_layers=1):
        super(CNNtoRNN, self).__init__()
        self.encoder = ResNetEncoder(embed_size=embed_size, use_attention=False)
        self.decoder = DecoderRNN(embed_size, hidden_size, vocab_size, num_layers)

    def forward(self, images, captions):
        features = self.encoder(images)
        outputs = self.decoder(features, captions)
        return outputs

# Evaluation metreic

In [ ]:
import collections
from nltk.translate.bleu_score import SmoothingFunction, corpus_bleu
import torch


def evaluate_bleu(model, val_df, vocab, images_dir, device, max_length=20):
    """Generates 1 caption per unique validation image and compares it

    against all 5 ground-truth references using corpus_bleu.
    """
    model.eval()

    # Group ground-truth references by image filename
    # Result structure: {'img1.jpg': [['a', 'dog', 'runs'], ['brown', 'dog', 'outside'], ...]}
    image_to_references = collections.defaultdict(list)
    for _, row in val_df.iterrows():
        img_id = row["image"]
        # Tokenize reference caption into a list of words
        ref_tokens = vocab.tokenize(row["caption"])
        image_to_references[img_id].append(ref_tokens)

    references_list = []
    hypotheses_list = []

    # Iterate through each UNIQUE image to generate a prediction
    unique_images = list(image_to_references.keys())

    with torch.no_grad():
        for img_id in unique_images:
            img_path = os.path.join(images_dir, img_id)

            # Load and preprocess image
            raw_img = Image.open(img_path).convert("RGB")
            img_tensor = transform(raw_img).unsqueeze(0).to(device)

            # --- Generate Caption (Greedy Search) ---
            features = model.encoder(img_tensor)
            states = None
            lstm_input = features.unsqueeze(1)

            predicted_tokens = []
            for _ in range(max_length):
                hiddens, states = model.decoder.lstm(lstm_input, states)
                output = model.decoder.linear(hiddens.squeeze(1))
                predicted_id = output.argmax(1).item()

                predicted_word = vocab.itos[predicted_id]
                if predicted_word == "<EOS>":
                    break

                predicted_tokens.append(predicted_word)
                lstm_input = model.decoder.embed(
                    torch.tensor([predicted_id]).to(device)
                ).unsqueeze(1)

            # Store the 5 references for this image and the single model prediction
            references_list.append(image_to_references[img_id])
            hypotheses_list.append(predicted_tokens)

    # Calculate Corpus BLEU scores (BLEU-1 to BLEU-4)
    # Smoothing technique prevents zero scores if 4-gram matches are sparse early in training
    smooth = SmoothingFunction().method1

    bleu1 = corpus_bleu(
        references_list,
        hypotheses_list,
        weights=(1.0, 0, 0, 0),
        smoothing_function=smooth,
    )
    bleu2 = corpus_bleu(
        references_list,
        hypotheses_list,
        weights=(0.5, 0.5, 0, 0),
        smoothing_function=smooth,
    )
    bleu3 = corpus_bleu(
        references_list,
        hypotheses_list,
        weights=(0.33, 0.33, 0.33, 0),
        smoothing_function=smooth,
    )
    bleu4 = corpus_bleu(
        references_list,
        hypotheses_list,
        weights=(0.25, 0.25, 0.25, 0.25),
        smoothing_function=smooth,
    )

    scores = {
        "BLEU-1": bleu1 * 100,
        "BLEU-2": bleu2 * 100,
        "BLEU-3": bleu3 * 100,
        "BLEU-4": bleu4 * 100,
    }

    print("\n=== Validation BLEU Scores ===")
    for metric, score in scores.items():
        print(f"{metric}: {score:.2f}")

    return scores

# Training

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim


EMBED_SIZE = 256
HIDDEN_SIZE = 512
VOCAB_SIZE = len(vocab)
NUM_LAYERS = 1
LEARNING_RATE = 0.0003
NUM_EPOCHS = 5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CNNtoRNN(EMBED_SIZE, HIDDEN_SIZE, VOCAB_SIZE, NUM_LAYERS).to(DEVICE)

# Ignore <PAD> tokens in loss calculation
pad_idx = vocab.stoi["<PAD>"]
criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
images_directory = os.path.join(DATASET_DIR, "Images")

# Training Loop
for epoch in range(NUM_EPOCHS):
    model.train()  # Ensure model is in train mode at start of epoch
    total_loss = 0.0

    for idx, (images, captions) in enumerate(train_loader):
        images = images.to(DEVICE)
        captions = captions.to(DEVICE)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images, captions[:, :-1])

        # Slice outputs to drop the final prediction timestep so length matches target
        outputs = outputs[:, :-1, :]

        # Reshape for CrossEntropyLoss
        targets = captions[:, 1:].contiguous().view(-1)
        outputs = outputs.reshape(-1, outputs.size(-1))

        loss = criterion(outputs, targets)

        # Backward pass & optimize
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if (idx + 1) % 100 == 0:
            print(
                f"Epoch [{epoch+1}/{NUM_EPOCHS}], Step [{idx+1}/{len(train_loader)}], Step Loss: {loss.item():.4f}"
            )

    # Calculate average loss after FULL epoch completes
    avg_loss = total_loss / len(train_loader)
    print(f"\n==> Epoch [{epoch+1}/{NUM_EPOCHS}] Average Loss: {avg_loss:.4f}")


# Save Model Weights
# Directory path setup
SAVE_DIR = "/mnt/e/linux_projects/MIA/task6/task6_2/weights"
os.makedirs(SAVE_DIR, exist_ok=True)  # Create folder if it doesn't exist
SAVE_PATH = os.path.join(SAVE_DIR, "enc_dec2.pth")

# Save state dictionary
torch.save(model.state_dict(), SAVE_PATH)
print(f"\nModel weights saved successfully to {SAVE_PATH}")

Epoch [1/5], Step [100/885], Step Loss: 5.2658
Epoch [1/5], Step [200/885], Step Loss: 4.6125
Epoch [1/5], Step [300/885], Step Loss: 4.6205
Epoch [1/5], Step [400/885], Step Loss: 4.4383
Epoch [1/5], Step [500/885], Step Loss: 4.2435
Epoch [1/5], Step [600/885], Step Loss: 4.3931
Epoch [1/5], Step [700/885], Step Loss: 4.2663
Epoch [1/5], Step [800/885], Step Loss: 4.3132

==> Epoch [1/5] Average Loss: 4.6938
Evaluating BLEU scores on Validation Set...

=== Validation BLEU Scores ===
BLEU-1: 41.54
BLEU-2: 11.32
BLEU-3: 0.62
BLEU-4: 0.14
Epoch [2/5], Step [100/885], Step Loss: 3.8420
Epoch [2/5], Step [200/885], Step Loss: 3.8158
Epoch [2/5], Step [300/885], Step Loss: 4.2280
Epoch [2/5], Step [400/885], Step Loss: 4.0465
Epoch [2/5], Step [500/885], Step Loss: 3.8828
Epoch [2/5], Step [600/885], Step Loss: 4.0705
Epoch [2/5], Step [700/885], Step Loss: 3.9126
Epoch [2/5], Step [800/885], Step Loss: 3.8226

==> Epoch [2/5] Average Loss: 4.0052
Evaluating BLEU scores on Validation Set..

In [ ]:
SAVE_PATH = "/mnt/e/linux_projects/MIA/task6/task6_2/weights/enc_dec1.pth"

model = CNNtoRNN(EMBED_SIZE, HIDDEN_SIZE, VOCAB_SIZE, NUM_LAYERS).to(DEVICE)

# Load the saved weights
model.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE))

model.eval()
print("Model loaded and ready for inference!")

NameError: name 'HIDDEN_SIZE' is not defined

# validation

In [ ]:
import collections
from nltk.translate.bleu_score import SmoothingFunction, corpus_bleu
import torch


def evaluate_bleu(model, val_df, vocab, images_dir, device, max_length=20):
    """Generates 1 caption per unique validation image and compares it

    against all 5 ground-truth references using corpus_bleu.
    """
    model.eval()

    # Group ground-truth references by image filename
    # Result structure: {'img1.jpg': [['a', 'dog', 'runs'], ['brown', 'dog', 'outside'], ...]}
    image_to_references = collections.defaultdict(list)
    for _, row in val_df.iterrows():
        img_id = row["image"]
        # Tokenize reference caption into a list of words
        ref_tokens = vocab.tokenize(row["caption"])
        image_to_references[img_id].append(ref_tokens)

    references_list = []
    hypotheses_list = []

    # Iterate through each UNIQUE image to generate a prediction
    unique_images = list(image_to_references.keys())

    with torch.no_grad():
        for img_id in unique_images:
            img_path = os.path.join(images_dir, img_id)

            # Load and preprocess image
            raw_img = Image.open(img_path).convert("RGB")
            img_tensor = transform(raw_img).unsqueeze(0).to(device)

            # --- Generate Caption (Greedy Search) ---
            features = model.encoder(img_tensor)
            states = None
            lstm_input = features.unsqueeze(1)

            predicted_tokens = []
            for _ in range(max_length):
                hiddens, states = model.decoder.lstm(lstm_input, states)
                output = model.decoder.linear(hiddens.squeeze(1))
                predicted_id = output.argmax(1).item()

                predicted_word = vocab.itos[predicted_id]
                if predicted_word == "<EOS>":
                    break

                predicted_tokens.append(predicted_word)
                lstm_input = model.decoder.embed(
                    torch.tensor([predicted_id]).to(device)
                ).unsqueeze(1)

            # Store the 5 references for this image and the single model prediction
            references_list.append(image_to_references[img_id])
            hypotheses_list.append(predicted_tokens)

    # Calculate Corpus BLEU scores (BLEU-1 to BLEU-4)
    # Smoothing technique prevents zero scores if 4-gram matches are sparse early in training
    smooth = SmoothingFunction().method1

    bleu1 = corpus_bleu(
        references_list,
        hypotheses_list,
        weights=(1.0, 0, 0, 0),
        smoothing_function=smooth,
    )
    bleu2 = corpus_bleu(
        references_list,
        hypotheses_list,
        weights=(0.5, 0.5, 0, 0),
        smoothing_function=smooth,
    )
    bleu3 = corpus_bleu(
        references_list,
        hypotheses_list,
        weights=(0.33, 0.33, 0.33, 0),
        smoothing_function=smooth,
    )
    bleu4 = corpus_bleu(
        references_list,
        hypotheses_list,
        weights=(0.25, 0.25, 0.25, 0.25),
        smoothing_function=smooth,
    )

    scores = {
        "BLEU-1": bleu1 * 100,
        "BLEU-2": bleu2 * 100,
        "BLEU-3": bleu3 * 100,
        "BLEU-4": bleu4 * 100,
    }

    print("\n=== Validation BLEU Scores ===")
    for metric, score in scores.items():
        print(f"{metric}: {score:.2f}")

    return scores

# validation

In [23]:
import torch


def evaluate_loss(model, val_loader, criterion, device):
    """Computes fast validation Cross-Entropy Loss on batch vectors (no text generation)."""
    model.eval()
    total_val_loss = 0.0

    with torch.no_grad():
        for images, captions in val_loader:
            images = images.to(device)
            captions = captions.to(device)

            # Forward pass
            outputs = model(images, captions[:, :-1])

            # Align timesteps
            outputs = outputs[:, :-1, :]
            targets = captions[:, 1:].contiguous().view(-1)
            outputs = outputs.reshape(-1, outputs.size(-1))

            loss = criterion(outputs, targets)
            total_val_loss += loss.item()

    return total_val_loss / len(val_loader)

In [ ]:
import collections
import os
import torch
from nltk.translate.bleu_score import SmoothingFunction, corpus_bleu
from PIL import Image

# Independent Single Image Caption Generator
def generate_caption_for_image(
    model, image_path, transform, vocab, device, max_length=20
):
    """Loads a single image from disk and autoregressively generates its caption."""
    model.eval()

    # Load and preprocess image
    raw_img = Image.open(image_path).convert("RGB")
    img_tensor = transform(raw_img).unsqueeze(0).to(device)

    predicted_tokens = []

    with torch.no_grad():
        # Pass image through ResNet encoder
        features = model.encoder(img_tensor)
        states = None

        # Prepend feature vector as the first sequence step (t0)
        lstm_input = features.unsqueeze(1)

        for _ in range(max_length):
            hiddens, states = model.decoder.lstm(lstm_input, states)
            output = model.decoder.linear(hiddens.squeeze(1))
            predicted_id = output.argmax(1).item()

            predicted_word = vocab.itos[predicted_id]
            if predicted_word == "<EOS>":
                break

            predicted_tokens.append(predicted_word)

            # Update input tensor with embedding of predicted word
            lstm_input = model.decoder.embed(
                torch.tensor([predicted_id]).to(device)
            ).unsqueeze(1)

    return " ".join(predicted_tokens)


# Independent BLEU Corpus Evaluator
def evaluate_bleu_corpus(
    model, val_df, vocab, transform, images_dir, device, max_length=20
):
    """Evaluates BLEU 1-4 scores across unique validation images against ground-truth references."""
    model.eval()

    # Group ground-truth references by image filename
    image_to_references = collections.defaultdict(list)
    for _, row in val_df.iterrows():
        img_id = row["image"]
        ref_tokens = vocab.tokenize(row["caption"])
        image_to_references[img_id].append(ref_tokens)

    references_list = []
    hypotheses_list = []

    unique_images = list(image_to_references.keys())

    for img_id in unique_images:
        img_path = os.path.join(images_dir, img_id)

        # Generate text tokens using the single image function logic
        caption_str = generate_caption_for_image( model, img_path, transform, vocab, device, max_length)
        print(caption_str)
        predicted_tokens = caption_str.split()

        references_list.append(image_to_references[img_id])
        hypotheses_list.append(predicted_tokens)

    # Compute BLEU Scores
    smooth = SmoothingFunction().method1
    scores = {
        "BLEU-1": corpus_bleu(
            references_list,
            hypotheses_list,
            weights=(1.0, 0, 0, 0),
            smoothing_function=smooth,
        )
        * 100,
        "BLEU-2": corpus_bleu(
            references_list,
            hypotheses_list,
            weights=(0.5, 0.5, 0, 0),
            smoothing_function=smooth,
        )
        * 100,
        "BLEU-3": corpus_bleu(
            references_list,
            hypotheses_list,
            weights=(0.33, 0.33, 0.33, 0),
            smoothing_function=smooth,
        )
        * 100,
        "BLEU-4": corpus_bleu(
            references_list,
            hypotheses_list,
            weights=(0.25, 0.25, 0.25, 0.25),
            smoothing_function=smooth,
        )
        * 100,
    }

    return scores

In [ ]:
# Create a PyTorch DataLoader for the validation set (Vector Validation)
val_dataset = Flickr8kDataset( df=val_df, images_dir=images_directory, 
                               vocab=vocab, transform=transform,)
val_loader = DataLoader( dataset=val_dataset, batch_size=32, 
                         shuffle=False, collate_fn=collate_fn,)



# Text-Based BLEU Validation
print("\nCalculating Final Validation BLEU Scores...")
bleu_scores = evaluate_bleu_corpus( model, val_df, vocab, transform, images_directory, DEVICE)

for metric, score in bleu_scores.items():
    print(f"{metric}: {score:.2f}")




Calculating Final Validation BLEU Scores...
a boy a on
a dog a
a dog a in
a boy a in red is a in
a dog a in
a girl a in is on beach
a dog a in is a on
a a car a in car a
a man a on bike a
a a horse a
two dogs a
a man a rock
a dog a in
a a is a in is a in
a dog a in field a
a man a on is a in of
a dog a
a dog a in field a
a dog a in field a
a boy a in is a
a boy a
a woman a and man a
a dog a
a dog a
a man a on bench a
a boy a in blue
a dog a in pool a
a man a with beard a
a dog a in pool a in pool a
a man a on mountain
a of is a in of on
a girl a in pink
a man a on bike a
a man a
a baby a in
a boy a in is a
a of with people in clothing a
a girl a
a boy a on
a man a with beard a
a man a in black a
a girl a
a of in of are a
a girl a in and shirt on sidewalk a
a boy a in blue
a dog a in
a boy a on skateboard a
a person a in is a
a boy a in
two men in suits
a man a and woman a
a a with is a in
a girl a in is a
a girl a in is on grass a
two children a and a and are in of are on floor a
a a 

TypeError: generate_caption_for_image() missing 5 required positional arguments: 'model', 'image_path', 'transform', 'vocab', and 'device'